# PARSeq OCR  ·  Augraphy  ·  IQA + U-Net Deblur
**Pipeline:**
1. Score all training images with BRISQUE (iqa-pytorch / pyiqa)
2. Take top-K best images → blur them → train U-Net deblur (residual learning)
3. PARSeq as OCR backbone (replaces TrOCR)
4. Augraphy document-degradation augmentations during training
5. Every image passes through the frozen U-Net before PARSeq
6. Hard-example mining + fine-tune

In [1]:
%pip install -q augraphy pyiqa timm albumentations editdistance

import os, cv2, torch, random, editdistance, re, warnings, shutil
import numpy as np, pandas as pd
import albumentations as A
from pathlib import Path
from typing import List, Optional, Tuple
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from tqdm.auto import tqdm
from datetime import datetime
import pyiqa
from augraphy import (
    InkBleed, NoiseTexturize, LightingGradient,
    DirtyDrum, AugraphyPipeline,
)
warnings.filterwarnings('ignore')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.8 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.9/218.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 102.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━

## 1. Rclone + Конфигурация

In [21]:
# ================= RCLONE SETUP =================
from kaggle_secrets import UserSecretsClient
import subprocess, base64

secrets = UserSecretsClient()
raw_conf = secrets.get_secret('RCLONE_CONF')
if '[gdrive]' in raw_conf and 'type =' in raw_conf:
    formatted_conf = raw_conf.replace(' type =', '\ntype =')\
                             .replace(' scope =', '\nscope =')\
                             .replace(' token =', '\ntoken =')\
                             .replace(' team_drive =', '\nteam_drive =')
else:
    formatted_conf = base64.b64decode(raw_conf.strip()).decode()

os.makedirs('/root/.config/rclone', exist_ok=True)
with open('/root/.config/rclone/rclone.conf', 'w') as f:
    f.write(formatted_conf)

if subprocess.run(['which', 'rclone'], capture_output=True).returncode != 0:
    os.system('curl https://rclone.org/install.sh | bash')

RCLONE_REMOTE = 'gdrive'

def rclone_upload(local_path, remote_subdir: str = ''):
    local_path = Path(local_path)
    dest = f'{RCLONE_REMOTE}:ocr_checkpoints/{remote_subdir}'.rstrip('/')
    cmd  = ['rclone', 'copy', str(local_path), dest,
            '--transfers=4', '-P'] if local_path.is_dir() else \
           ['rclone', 'copy', str(local_path), dest, '-P']
    res  = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f'[rclone ERROR] {res.stderr}')
    else:
        print(f'[rclone] Загружено: {local_path} -> {dest}')

# ================= НАСТРОЙКИ =================
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

CKPT_DIR = Path(f'/kaggle/working/ocr_checkpoints_{timestamp}')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = '/kaggle/working/submission.csv'

DATA_DIR = Path('/kaggle/input/datasets/denccchicck/dl-lab-4-ocr-custom-dataset')
NESTED_DATA_DIR = DATA_DIR / 'dl-lab-4-ocr-custom-dataset-v5'
if not NESTED_DATA_DIR.exists():
    NESTED_DATA_DIR = DATA_DIR
    print(f'[INFO] Используем DATA_DIR напрямую: {NESTED_DATA_DIR}')
else:
    print(f'[INFO] NESTED_DATA_DIR: {NESTED_DATA_DIR}')

# PARSeq input resolution
IMG_H, IMG_W = 32, 128

BATCH_SIZE  = 32
NUM_EPOCHS  = 10
LR          = 1e-4
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0

# IQA / Deblur
IQA_TOP_K   = 1000   # сколько лучших изображений берём для обучения U-Net
UNET_EPOCHS = 5
UNET_LR     = 1e-3
UNET_H, UNET_W = 64, 256  # разрешение при обучении U-Net

TRAIN_IMG_DIR = NESTED_DATA_DIR / 'train' / 'train'
VAL_IMG_DIR   = NESTED_DATA_DIR / 'val'   / 'val'
TEST_IMG_DIR  = NESTED_DATA_DIR / 'test'  / 'test'

seed = 42
random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed); torch.backends.cudnn.benchmark = True

print(f'CKPT_DIR : {CKPT_DIR}')
print(f'DEVICE   : {DEVICE}  |  '
      f'{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')


[INFO] NESTED_DATA_DIR: /kaggle/input/datasets/denccchicck/dl-lab-4-ocr-custom-dataset/dl-lab-4-ocr-custom-dataset-v5
CKPT_DIR : /kaggle/working/ocr_checkpoints_20260501_073439
DEVICE   : cuda  |  Tesla T4


## 2. Данные

In [22]:
train_df = pd.read_csv(NESTED_DATA_DIR / 'train.csv')
val_df   = pd.read_csv(NESTED_DATA_DIR / 'val.csv')
test_df  = pd.read_csv(NESTED_DATA_DIR / 'sample_submission.csv')

# Приводим Price к строке — PARSeq декодирует текст
train_df['Price'] = train_df['Price'].astype(str)
val_df['Price']   = val_df['Price'].astype(str)

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
print(train_df.head())


Train: 33500  Val: 15050  Test: 3762
         Filename Price
0   00000_871.jpg   871
1  00001_1418.jpg  1418
2    00002_15.jpg    15
3   00003_429.jpg   429
4  00004_1678.jpg  1678


## 3. IQA — отбираем лучшие изображения для обучения U-Net

BRISQUE: чем **ниже** — тем лучше качество (меньше артефактов).  
Берём `IQA_TOP_K` изображений с наименьшим score → деградируем их блюром → обучаем U-Net вернуть исходное качество.

In [23]:
def get_img_paths(df: pd.DataFrame, base_dir: Path) -> List[Path]:
    paths = []
    for _, row in df.iterrows():
        folder = row.get('img_folder', '')
        paths.append(base_dir / folder / row['Filename'])
    return paths


train_img_paths = [TRAIN_IMG_DIR / fname for fname in train_df['Filename']]

print('Scoring training images with BRISQUE (pyiqa)…')
_brisque = pyiqa.create_metric('brisque', device=DEVICE)
_to_t    = transforms.ToTensor()

def brisque_score(img_paths: List[Path]) -> np.ndarray:
    scores = []
    for p in tqdm(img_paths, desc='IQA-BRISQUE'):
        try:
            img = cv2.imdecode(np.fromfile(str(p), dtype=np.uint8), cv2.IMREAD_COLOR)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            t   = _to_t(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                s = _brisque(t).item()
        except Exception:
            s = 1e6   # сломанные → в конец
        scores.append(s)
    return np.array(scores)


iqa_scores  = brisque_score(train_img_paths)
sorted_idx  = np.argsort(iqa_scores)           # ascending: лучшие первые
top_k_paths = [train_img_paths[i] for i in sorted_idx[:IQA_TOP_K]]

print(f'Отобрано топ-{IQA_TOP_K} изображений.')
print(f'BRISQUE range:  [{iqa_scores.min():.1f}, {iqa_scores.max():.1f}]')
print(f'Top-K range:    [{iqa_scores[sorted_idx[0]]:.1f}, '
      f'{iqa_scores[sorted_idx[IQA_TOP_K-1]]:.1f}]')


Scoring training images with BRISQUE (pyiqa)…


IQA-BRISQUE:   0%|          | 0/33500 [00:00<?, ?it/s]

Отобрано топ-1000 изображений.
BRISQUE range:  [1.4, 138.4]
Top-K range:    [1.4, 29.2]


## 4. Лёгкий U-Net для деблюринга

**Residual learning:** `output = clamp(input + U-Net(input), 0, 1)`  
U-Net предсказывает остаток, а не саму картинку — это стабильнее для небольших датасетов.

In [24]:
class _DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class DeblurUNet(nn.Module):
    """U-Net с residual learning для деблюринга текстовых / ценниковых изображений.
    Выходной residual ∈ [-1, 1].  Итог: clamp(input + residual, 0, 1).
    """
    C = [3, 32, 64, 128, 256]

    def __init__(self):
        super().__init__()
        C = self.C
        # Encoder
        self.enc0 = _DoubleConv(C[0], C[1])
        self.enc1 = _DoubleConv(C[1], C[2])
        self.enc2 = _DoubleConv(C[2], C[3])
        self.bot  = _DoubleConv(C[3], C[4])
        # Decoder
        self.up2  = nn.ConvTranspose2d(C[4], C[3], 2, stride=2)
        self.dec2 = _DoubleConv(C[3]*2, C[3])
        self.up1  = nn.ConvTranspose2d(C[3], C[2], 2, stride=2)
        self.dec1 = _DoubleConv(C[2]*2, C[2])
        self.up0  = nn.ConvTranspose2d(C[2], C[1], 2, stride=2)
        self.dec0 = _DoubleConv(C[1]*2, C[1])
        self.head = nn.Conv2d(C[1], 3, 1)
        self.pool = nn.MaxPool2d(2)
    
    def forward(self, x):
        # Encoder
        e0 = self.enc0(x)
        e1 = self.enc1(self.pool(e0))
        e2 = self.enc2(self.pool(e1))
        b  = self.bot(self.pool(e2))
    
        # Decoder 2
        u2 = self.up2(b)
        e2 = e2[:, :, :u2.size(2), :u2.size(3)] # Обрезаем лишний пиксель, если он есть
        d2 = self.dec2(torch.cat([u2, e2], 1))
    
        # Decoder 1
        u1 = self.up1(d2)
        e1 = e1[:, :, :u1.size(2), :u1.size(3)] # <--- Ошибка была здесь
        d1 = self.dec1(torch.cat([u1, e1], 1))
    
        # Decoder 0
        u0 = self.up0(d1)
        e0 = e0[:, :, :u0.size(2), :u0.size(3)]
        d0 = self.dec0(torch.cat([u0, e0], 1))
    
        return torch.tanh(self.head(d0))
    # def forward(self, x: torch.Tensor) -> torch.Tensor:
    #     e0 = self.enc0(x)
    #     e1 = self.enc1(self.pool(e0))
    #     e2 = self.enc2(self.pool(e1))
    #     b  = self.bot(self.pool(e2))
    # # Пример для d2:
    #     up_b = self.up2(b)
    #     # Если размеры не совпали с e2, подгоняем up_b
    #     if up_b.shape != e2.shape:
    #         up_b = F.interpolate(up_b, size=(e2.shape[2], e2.shape[3]), mode='bilinear')
    #     d2 = self.dec2(torch.cat([up_b, e2], 1))
    #     d2 = self.dec2(torch.cat([self.up2(b),  e2], 1))
    #     d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
    #     d0 = self.dec0(torch.cat([self.up0(d1), e0], 1))
    #     residual = torch.tanh(self.head(d0))    # [-1, 1]
    #     # blur + deblur = деблюренная оценка изображения
    #     return (x + residual).clamp(0.0, 1.0)


print(f'DeblurUNet params: {sum(p.numel() for p in DeblurUNet().parameters()):,}')


DeblurUNet params: 1,927,075


## 5. Обучение U-Net на топ-K изображениях

In [25]:
import torch
print(f"Карта: {torch.cuda.get_device_name(0)}")
print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
print(f"Версия PyTorch: {torch.__version__}")

Карта: Tesla T4
Compute Capability: (7, 5)
Версия PyTorch: 2.10.0+cu128


In [26]:
class DeblurDataset(Dataset):
    """Генерирует пары (blurry, sharp) на лету."""
    KERNELS = [3, 5, 7, 9]

    def __init__(self, img_paths: List[Path], size: Tuple[int, int] = (64, 256)):
        self.paths  = img_paths
        self.h, self.w = size
        self.to_t   = transforms.ToTensor()

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx: int):
        img = cv2.imdecode(np.fromfile(str(self.paths[idx]), dtype=np.uint8),
                           cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.w, self.h), interpolation=cv2.INTER_CUBIC)
        sharp  = self.to_t(img)                                    # [0,1]
        k      = int(np.random.choice(self.KERNELS))
        blurry = self.to_t(cv2.GaussianBlur(img, (k, k), 0))
        return blurry, sharp


def train_deblur_unet(
    paths: List[Path],
    device,
    epochs: int   = UNET_EPOCHS,
    lr:     float = UNET_LR,
    save_path: Optional[Path] = None,
) -> DeblurUNet:
    ds     = DeblurDataset(paths, size=(UNET_H, UNET_W))
    loader = DataLoader(ds, batch_size=16, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True)
    net    = DeblurUNet().to(device)
    opt    = torch.optim.Adam(net.parameters(), lr=lr)
    scaler = torch.amp.GradScaler('cuda')

    for ep in range(1, epochs + 1):
        net.train()
        ep_loss = 0.0
        pbar = tqdm(loader, desc=f'U-Net Deblur  ep {ep}/{epochs}')
        for blurry, sharp in pbar:
            blurry, sharp = blurry.to(device), sharp.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                pred = net(blurry)
                loss = F.mse_loss(pred, sharp) + 0.1 * F.l1_loss(pred, sharp)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            ep_loss += loss.item()
            pbar.set_postfix(loss=f'{ep_loss / len(pbar):.5f}')

    if save_path:
        torch.save(net.state_dict(), save_path)
        print(f'Deblur U-Net сохранён → {save_path}')
    net.eval()
    return net


deblur_model = train_deblur_unet(
    top_k_paths, DEVICE,
    save_path=CKPT_DIR / 'deblur_unet.pt',
)
print('Deblur model готов.')


U-Net Deblur  ep 1/5:   0%|          | 0/63 [00:00<?, ?it/s]

U-Net Deblur  ep 2/5:   0%|          | 0/63 [00:00<?, ?it/s]

U-Net Deblur  ep 3/5:   0%|          | 0/63 [00:00<?, ?it/s]

U-Net Deblur  ep 4/5:   0%|          | 0/63 [00:00<?, ?it/s]

U-Net Deblur  ep 5/5:   0%|          | 0/63 [00:00<?, ?it/s]

Deblur U-Net сохранён → /kaggle/working/ocr_checkpoints_20260501_073439/deblur_unet.pt
Deblur model готов.


## 6. PARSeq — загрузка модели

PARSeq (Parseq, baudm/parseq) — state-of-the-art STR модель.  
Загружается через `torch.hub`, веса скачиваются автоматически.  
Входной размер: **32×128** (H×W).  
Токенизатор встроен в модель (`model.tokenizer`).  
Для обучения используется `model.forward_logits_loss(images, labels)`,  
для инференса — `model(images)` + `model.tokenizer.decode(probs)`.

In [27]:
print('Загружаем PARSeq…')
parseq = torch.hub.load(
    'baudm/parseq', 'parseq',
    pretrained=True,
    trust_repo=True,
).to(DEVICE)
print('PARSeq загружен.')
print(f'PARSeq params: {sum(p.numel() for p in parseq.parameters()):,}')


def preprocess_for_parseq(img_rgb: np.ndarray) -> torch.Tensor:
    """uint8 HxWxC → Tensor [3, 32, 128] в диапазоне [-1, 1]."""
    img = cv2.resize(img_rgb, (IMG_W, IMG_H), interpolation=cv2.INTER_CUBIC)
    t   = transforms.ToTensor()(img)        # [0, 1]
    return transforms.Normalize(0.5, 0.5)(t) # [-1, 1]


Загружаем PARSeq…


Using cache found in /root/.cache/torch/hub/baudm_parseq_main


PARSeq загружен.
PARSeq params: 23,832,671


## 7. Augraphy — документные деградации

Augraphy моделирует физические деградации: кровотечение чернил, освещение, грязь, складки.  
Применяется в pipeline'е датасета с вероятностью 0.5.

In [28]:
def build_augraphy_pipeline() -> AugraphyPipeline:
    return AugraphyPipeline(
        ink_phase=[
            InkBleed(
                intensity_range=(0.1, 0.3),
                kernel_size=(7, 7),
                severity=(0.4, 0.6),
                p=0.4,
            ),
        ],
        paper_phase=[
            NoiseTexturize(
                sigma_range=(3, 10),
                turbulence_range=(2, 5),
                p=0.35,
            ),
            LightingGradient(
                max_brightness=255,
                min_brightness=128,
                mode='gaussian',
                p=0.3,
            ),
        ],
        post_phase=[
            DirtyDrum(
                line_width_range=(1, 3),
                line_concentration=0.05,
                direction=0,
                noise_intensity=0.5,
                noise_value=(64, 224),
                ksize=(3, 3),
                sigmaX=0,
                p=0.3,
            )
        ],
    )


_augraphy = build_augraphy_pipeline()


def apply_augraphy(img_rgb: np.ndarray) -> np.ndarray:
    """Augraphy работает в BGR, конвертируем туда-обратно."""
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    aug = _augraphy(bgr)
    return cv2.cvtColor(aug, cv2.COLOR_BGR2RGB)


print('Augraphy pipeline готов.')


Augraphy pipeline готов.


## 8. Dataset

**Порядок препроцессинга на каждой картинке:**
1. (train only, p=0.5) Augraphy document degradation
2. Albumentations (brightness, contrast, rotation)
3. U-Net деблюр: `sharp_est = blur + U-Net(blur)`
4. Resize → 32×128 + Normalize [-1,1] для PARSeq

In [29]:
class PriceDataset(Dataset):
    def __init__(
        self,
        df:           pd.DataFrame,
        img_dir:      Path,
        deblur_model: Optional[DeblurUNet] = None,
        augment:      bool  = False,
        is_test:      bool  = False,
    ):
        self.df     = df.reset_index(drop=True)
        self.imgdir = Path(img_dir)
        # self.deblur = deblur_model
        self.augment= augment
        self.is_test= is_test
        self._to_t  = transforms.ToTensor()

        self.albu = A.Compose([
            A.RandomBrightnessContrast(
                brightness_limit=0.25, contrast_limit=0.25, p=0.5),
            A.Rotate(limit=3, border_mode=cv2.BORDER_REPLICATE, p=0.3),
            A.GaussianBlur(blur_limit=(3, 5), p=0.15),
            A.ImageCompression(quality_lower=70, quality_upper=95, p=0.2),
            A.HueSaturationValue(
                hue_shift_limit=15, sat_shift_limit=20, val_shift_limit=10, p=0.4),
        ]) if augment else None

    # ── helpers ─────────────────────────────────────────────


    @torch.no_grad()
    def _run_deblur(self, img_rgb: np.ndarray) -> np.ndarray:
        t   = self._to_t(img_rgb).unsqueeze(0)
        dev = next(self.deblur.parameters()).device
        out = self.deblur(t.to(dev)).squeeze(0).cpu().clamp(0, 1)  # .cpu() обязателен
        return (out.numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
    # def _run_deblur(self, img_rgb: np.ndarray) -> np.ndarray:
    #     dev = next(self.deblur.parameters()).device
    #     t = self._to_t(img_rgb).unsqueeze(0) # [1, 3, H, W]
        
    #     # Добавляем отступы до ближайшего числа, кратного 32 (на всякий случай)
    #     h, w = t.shape[2], t.shape[3]
    #     new_h = (h + 31) // 32 * 32
    #     new_w = (w + 31) // 32 * 32
        
    #     pad_h = new_h - h
    #     pad_w = new_w - w
        
    #     # Паддинг (справа и снизу)
    #     t_padded = F.pad(t, (0, pad_w, 0, pad_h))
        
    #     # Прогон через модель
    #     out_padded = self.deblur(t_padded.to(dev))
        
    #     # Обрезаем обратно до оригинального размера
    #     out = out_padded[:, :, :h, :w]
    # #     """Пропускаем через U-Net. Возвращает uint8 RGB."""
    # #     t   = self._to_t(img_rgb).unsqueeze(0)
    # #     out = self.deblur(t.to(dev)).squeeze(0).cpu().clamp(0, 1)
    #     return (out.numpy().transpose(1, 2, 0) * 255).astype(np.uint8)

    
    def _preprocess(self, img_rgb: np.ndarray) -> torch.Tensor:
        # 1. Augraphy
        if self.augment and np.random.rand() < 0.5:
            try:
                img_rgb = apply_augraphy(img_rgb)
            except Exception:
                pass

        # 2. Albumentations
        if self.albu is not None:
            img_rgb = self.albu(image=img_rgb)['image']

        # # 3. U-Net deblur  (blur + deblur → чёткая оценка)
        # if self.deblur is not None:
        #     img_rgb = self._run_deblur(img_rgb)

        # 4. Resize + normalise для PARSeq
        return preprocess_for_parseq(img_rgb)

    # ── Dataset API ─────────────────────────────────────────

    def __len__(self): return len(self.df)

    def __getitem__(self, idx: int):
        row      = self.df.iloc[idx]
        img_path = self.imgdir / row['Filename']   # просто imgdir/filename, без folder
    
        img = cv2.imdecode(np.fromfile(str(img_path), dtype=np.uint8),
                           cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pv  = self._preprocess(img)
    
        if self.is_test:
            return pv, row['Filename']
        return pv, str(row['Price'])


In [30]:
train_ds = PriceDataset(train_df, TRAIN_IMG_DIR, augment=True)
val_ds   = PriceDataset(val_df,   VAL_IMG_DIR,   augment=False)
test_ds  = PriceDataset(test_df,  TEST_IMG_DIR,  augment=False, is_test=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)}  '
      f'Val batches: {len(val_loader)}  '
      f'Test batches: {len(test_loader)}')


Train batches: 1047  Val batches: 471  Test batches: 59


## 9. Обучение PARSeq

In [31]:
# Диагностика — запусти отдельной ячейкой
import pandas as pd
from pathlib import Path

print("=== val_df columns ===")
print(val_df.columns.tolist())
print(val_df.head(3))
print()
print("=== train_df columns ===")
print(train_df.columns.tolist())
print(train_df.head(3))
print()

# Что реально есть в NESTED_DATA_DIR
print("=== Структура директорий ===")
for p in sorted(NESTED_DATA_DIR.iterdir()):
    print(p)
    if p.is_dir():
        sub = list(p.iterdir())[:3]
        for s in sub:
            print(f"  {s}")

=== val_df columns ===
['Filename', 'Price']
                                       Filename Price
0   2_10-B4-1D-E0-2F-E4_2026-01-22-13-59-57.jpg    66
1  21_D0-CF-13-22-25-50_2026-01-22-14-03-26.jpg    35
2  22_D0-CF-13-3A-86-C8_2026-01-22-22-12-38.jpg    57

=== train_df columns ===
['Filename', 'Price']
         Filename Price
0   00000_871.jpg   871
1  00001_1418.jpg  1418
2    00002_15.jpg    15

=== Структура директорий ===
/kaggle/input/datasets/denccchicck/dl-lab-4-ocr-custom-dataset/dl-lab-4-ocr-custom-dataset-v5/sample_submission.csv
/kaggle/input/datasets/denccchicck/dl-lab-4-ocr-custom-dataset/dl-lab-4-ocr-custom-dataset-v5/test
  /kaggle/input/datasets/denccchicck/dl-lab-4-ocr-custom-dataset/dl-lab-4-ocr-custom-dataset-v5/test/test
/kaggle/input/datasets/denccchicck/dl-lab-4-ocr-custom-dataset/dl-lab-4-ocr-custom-dataset-v5/test.csv
/kaggle/input/datasets/denccchicck/dl-lab-4-ocr-custom-dataset/dl-lab-4-ocr-custom-dataset-v5/train
  /kaggle/input/datasets/denccchicck/dl-l

In [32]:
import torch.multiprocessing as mp
mp.set_start_method('spawn', force=True)

def train_epoch(model, loader, optimizer, scaler, device) -> float:
    model.train()
    deblur_model.eval()
    total = 0.0
    pbar = tqdm(loader, desc='  train', leave=False)
    
    for images, labels in pbar:
        images = images.to(device)
        
        # Деблюр батчем — один forward на весь батч, быстро
        with torch.no_grad():
            images = deblur_model(images)   # уже на GPU, .cpu() не нужен
        
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            _, loss, _ = model.forward_logits_loss(images, list(labels))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
        pbar.set_postfix(loss=f'{total / len(pbar):.4f}')
    return total / len(loader)
# def train_epoch(model, loader, optimizer, scaler, device) -> float:
#     model.train()
#     total = 0.0
#     pbar  = tqdm(loader, desc='  train', leave=False)
#     for images, labels in pbar:
#         images = images.to(device)
#         optimizer.zero_grad(set_to_none=True)
#         with torch.amp.autocast('cuda'):
#             # forward_logits_loss(images, labels: List[str]) -> (logits, loss, n)
#             _, loss, _ = model.forward_logits_loss(images, list(labels))
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         total += loss.item()
#         pbar.set_postfix(loss=f'{total / len(pbar):.4f}')
#     return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device) -> Tuple[float, float]:
    model.eval()
    deblur_model.eval()
    total_loss = 0.0
    correct = total = 0
    pbar = tqdm(loader, desc='  val  ', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        images = deblur_model(images)       # добавить здесь тоже
        with torch.amp.autocast('cuda'):
            logits, loss, _ = model.forward_logits_loss(images, list(labels))
        total_loss += loss.item()
        probs = logits.softmax(-1)
        preds, _ = model.tokenizer.decode(probs)
        for pred, lbl in zip(preds, labels):
            correct += (re.sub(r'\D', '', pred) == re.sub(r'\D', '', str(lbl)))
            total   += 1
    return total_loss / len(loader), correct / total
# def eval_epoch(model, loader, device) -> Tuple[float, float]:
#     model.eval()
#     total_loss = 0.0
#     correct = total = 0
#     pbar = tqdm(loader, desc='  val  ', leave=False)
#     for images, labels in pbar:
#         images = images.to(device)
#         with torch.amp.autocast('cuda'):
#             logits, loss, _ = model.forward_logits_loss(images, list(labels))
#         total_loss += loss.item()
#         # Decode predictions
#         probs = logits.softmax(-1)
#         preds, _ = model.tokenizer.decode(probs)
#         for pred, lbl in zip(preds, labels):
#             correct += (re.sub(r'\D', '', pred) == re.sub(r'\D', '', str(lbl)))
#             total   += 1
#     return total_loss / len(loader), correct / total


optimizer = torch.optim.AdamW(parseq.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=LR / 10)
scaler    = torch.amp.GradScaler('cuda')

best_val_loss = float('inf')
print(f'Обучаем PARSeq {NUM_EPOCHS} эпох…\n')

for ep in range(1, NUM_EPOCHS + 1):
    tr = train_epoch(parseq, train_loader, optimizer, scaler, DEVICE)
    vl, acc = eval_epoch(parseq, val_loader, DEVICE)
    scheduler.step()

    marker = ''
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(parseq.state_dict(), CKPT_DIR / 'parseq_best.pt')
        marker = '  ← saved'

    print(f'Ep {ep:02d}/{NUM_EPOCHS}  '
          f'tr={tr:.4f}  val={vl:.4f}  acc={acc:.4f}{marker}')

print('\nОбучение завершено.')
rclone_upload(CKPT_DIR / 'parseq_best.pt', f'run_{timestamp}')


Обучаем PARSeq 10 эпох…



  train:   0%|          | 0/1047 [00:00<?, ?it/s]

  val  :   0%|          | 0/471 [00:00<?, ?it/s]

Ep 01/10  tr=0.2838  val=0.0264  acc=0.9746  ← saved


  train:   0%|          | 0/1047 [00:00<?, ?it/s]

...

  val  :   0%|          | 0/471 [00:00<?, ?it/s]

Ep 09/10  tr=0.0290  val=0.0254  acc=0.9862


  train:   0%|          | 0/1047 [00:00<?, ?it/s]

  val  :   0%|          | 0/471 [00:00<?, ?it/s]

Ep 10/10  tr=0.0276  val=0.0245  acc=0.9847

Обучение завершено.
[rclone] Загружено: /kaggle/working/ocr_checkpoints_20260501_073439/parseq_best.pt -> gdrive:ocr_checkpoints/run_20260501_073439


## 10. Инференс

In [33]:
# Загружаем лучшие веса
parseq.load_state_dict(
    torch.load(CKPT_DIR / 'parseq_best.pt', map_location=DEVICE))
parseq.eval()


@torch.no_grad()
def predict(model, loader, device) -> pd.DataFrame:
    rows = []
    for images, fnames in tqdm(loader, desc='Predict'):
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            logits = model(images)    # [B, T, V]
        probs = logits.softmax(-1)
        preds, _ = model.tokenizer.decode(probs)
        for fname, pred in zip(fnames, preds):
            digits = re.sub(r'\D', '', pred)
            rows.append({'Filename': fname,
                         'Price':    int(digits) if digits else 0})
    return pd.DataFrame(rows)


results_df = predict(parseq, test_loader, DEVICE)
results_df.to_csv(OUTPUT_CSV, index=False)
print(f'Сабмишн → {OUTPUT_CSV}')
rclone_upload(OUTPUT_CSV, f'run_{timestamp}')


Predict:   0%|          | 0/59 [00:00<?, ?it/s]

Сабмишн → /kaggle/working/submission.csv
[rclone] Загружено: /kaggle/working/submission.csv -> gdrive:ocr_checkpoints/run_20260501_073439


## 11. Hard Example Mining + Fine-tune

In [35]:
# Скорим валидацию — ищем, где модель ошибается
@torch.no_grad()
def score_predictions(model, df, img_dir, deblur, device) -> pd.DataFrame:
    ds     = PriceDataset(df, img_dir, augment=False)  # без deblur_model
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS)
    rows   = []
    for images, labels in tqdm(loader, desc='Scoring val'):
        images = images.to(device)
        images = deblur(images)                        # деблюр здесь
        with torch.amp.autocast('cuda'):
            logits = model(images)
        probs = logits.softmax(-1)
        preds, _ = model.tokenizer.decode(probs)
        for pred, lbl in zip(preds, labels):
            p = re.sub(r'\D', '', pred)
            l = re.sub(r'\D', '', str(lbl))
            rows.append({'pred': p, 'label': l, 'correct': p == l})
    return pd.DataFrame(rows)

# Скорим валидацию
score_df = score_predictions(parseq, val_df, VAL_IMG_DIR, deblur_model, DEVICE)
hard_mask = ~score_df['correct'].values
hard_val  = val_df[hard_mask].reset_index(drop=True)
print(f'Hard examples: {hard_mask.sum()} / {len(val_df)}')

# hard_val берётся из val — добавляем img_folder чтобы PriceDataset знал путь
# Но раз мы передаём VAL_IMG_DIR напрямую, нужен отдельный датасет для hard_val
hard_ds = PriceDataset(hard_val, VAL_IMG_DIR, augment=True)

# train тоже с правильной папкой
base_ds = PriceDataset(train_df, TRAIN_IMG_DIR, augment=True)

# Микс через ConcatDataset — не нужно мержить df с разными img_dir
from torch.utils.data import ConcatDataset
ft_ds = ConcatDataset([base_ds, hard_ds, hard_ds])  # hard 2x oversampling

ft_loader = DataLoader(ft_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=True)
# score_df  = score_predictions(parseq, val_df, NESTED_DATA_DIR, deblur_model, DEVICE)
# hard_mask = ~score_df['correct'].values
# hard_val  = val_df[hard_mask].reset_index(drop=True)
# print(f'Hard examples: {hard_mask.sum()} / {len(val_df)}')

# # Микс: весь train + 2× hard val
# mix_df = pd.concat([train_df, hard_val, hard_val], ignore_index=True)
# print(f'Fine-tune mix: {len(mix_df)} samples')

# ft_ds     = PriceDataset(mix_df, NESTED_DATA_DIR,
#                           deblur_model=deblur_model, augment=True)
# ft_loader = DataLoader(ft_ds, batch_size=BATCH_SIZE, shuffle=True,
#                         num_workers=NUM_WORKERS, pin_memory=True)

ft_opt    = torch.optim.AdamW(parseq.parameters(), lr=LR / 10, weight_decay=0.01)
ft_scaler = torch.amp.GradScaler('cuda')
FT_EPOCHS = 3

print(f'Fine-tuning {FT_EPOCHS} эпохи…')
for ep in range(1, FT_EPOCHS + 1):
    tr = train_epoch(parseq, ft_loader, ft_opt, ft_scaler, DEVICE)
    vl, acc = eval_epoch(parseq, val_loader, DEVICE)
    print(f'FT ep {ep}  tr={tr:.4f}  val={vl:.4f}  acc={acc:.4f}')

results_ft = predict(parseq, test_loader, DEVICE)
ft_csv = '/kaggle/working/submission_parseq_ft.csv'
results_ft.to_csv(ft_csv, index=False)
print(f'Fine-tuned сабмишн → {ft_csv}')
rclone_upload(ft_csv, f'run_{timestamp}')


Scoring val:   0%|          | 0/471 [00:00<?, ?it/s]

Hard examples: 245 / 15050
Fine-tuning 3 эпохи…


  train:   0%|          | 0/1063 [00:00<?, ?it/s]

  val  :   0%|          | 0/471 [00:00<?, ?it/s]

FT ep 1  tr=0.0424  val=0.0095  acc=0.9924


...

  train:   0%|          | 0/1063 [00:00<?, ?it/s]

  val  :   0%|          | 0/471 [00:00<?, ?it/s]

FT ep 3  tr=0.0364  val=0.0077  acc=0.9937


Predict:   0%|          | 0/59 [00:00<?, ?it/s]

Fine-tuned сабмишн → /kaggle/working/submission_parseq_ft.csv
[rclone] Загружено: /kaggle/working/submission_parseq_ft.csv -> gdrive:ocr_checkpoints/run_20260501_073439


## 12. Анализ расхождений base ↔ ft

In [36]:
df_base = pd.read_csv(OUTPUT_CSV)
df_ft   = pd.read_csv(ft_csv)
df_comp = pd.merge(df_base, df_ft, on='Filename', suffixes=('_base', '_ft'))
diff_df = df_comp[df_comp['Price_base'] != df_comp['Price_ft']]

print(f'Всего: {len(df_comp)}  |  Расхождений base↔ft: {len(diff_df)}')

DISC_DIR = CKPT_DIR / 'discrepancies'
DISC_DIR.mkdir(exist_ok=True)
diff_df.to_csv(CKPT_DIR / 'comparison_summary.csv', index=False)

test_img_dir = NESTED_DATA_DIR / 'test' / 'test'
copied = 0
for _, row in tqdm(diff_df.iterrows(), total=len(diff_df),
                   desc='Copying discrepancies'):
    src = test_img_dir / row['Filename']
    if src.exists():
        dst = DISC_DIR / (
            f"base_{row['Price_base']}_ft_{row['Price_ft']}_{row['Filename']}")
        shutil.copy(str(src), str(dst))
        copied += 1

print(f'Скопировано {copied} изображений → {DISC_DIR}')
rclone_upload(CKPT_DIR / 'comparison_summary.csv', f'run_{timestamp}')
rclone_upload(DISC_DIR, f'run_{timestamp}/discrepancies')


Всего: 3762  |  Расхождений base↔ft: 10


Copying discrepancies:   0%|          | 0/10 [00:00<?, ?it/s]

Скопировано 10 изображений → /kaggle/working/ocr_checkpoints_20260501_073439/discrepancies
[rclone] Загружено: /kaggle/working/ocr_checkpoints_20260501_073439/comparison_summary.csv -> gdrive:ocr_checkpoints/run_20260501_073439
[rclone] Загружено: /kaggle/working/ocr_checkpoints_20260501_073439/discrepancies -> gdrive:ocr_checkpoints/run_20260501_073439/discrepancies
